# 02 — WBGT Processing: Daily WBGT → Average Annual Productivity Loss

For a defined epoch (year range), this notebook:
1. Iterates over all daily WBGT GeoTIFFs
2. Applies the exposure-response function (WBGT → productivity loss)
3. Accumulates and averages to produce a single **mean annual productivity loss** raster
4. Saves the output to `data/processed/`

**Before running:** ensure `scripts/productivity.py` has `WBGT_THRESHOLDS` and `PRODUCTIVITY_VALUES` defined.

In [ ]:
import sys
from pathlib import Path
from datetime import date, timedelta

import numpy as np
import rasterio
import yaml
import requests
from tqdm import tqdm

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from scripts.productivity import wbgt_to_productivity_loss

with open(ROOT / 'config' / 'config.yaml') as f:
    config = yaml.safe_load(f)

DATA = ROOT / 'data'
WBGT_DIR = DATA / 'wbgt'
PROCESSED_DIR = DATA / 'processed'
PROCESSED_DIR.mkdir(exist_ok=True)
print('Ready.')

## Configuration — set epoch here

In [ ]:
# Choose epoch: any start/end year within the available data range
EPOCH_START = config['wbgt']['start_year']   # e.g. 1983
EPOCH_END   = config['wbgt']['end_year']     # e.g. 2016

# Output filename for this epoch
output_path = PROCESSED_DIR / f'productivity_loss_{EPOCH_START}_{EPOCH_END}.tif'
print(f'Epoch: {EPOCH_START}–{EPOCH_END}')
print(f'Output: {output_path}')

## (Optional) Download WBGT files

Skip this cell if files are already downloaded.

In [ ]:
def iter_dates(start_year: int, end_year: int):
    """Yield every date from start_year-01-01 to end_year-12-31."""
    d = date(start_year, 1, 1)
    end = date(end_year, 12, 31)
    while d <= end:
        yield d
        d += timedelta(days=1)


def download_wbgt(start_year: int, end_year: int, dest_dir: Path, skip_existing: bool = True):
    """Download daily WBGT GeoTIFFs from CHIRTS-ERA5."""
    url_template = config['wbgt']['url_template']
    dest_dir.mkdir(parents=True, exist_ok=True)

    dates = list(iter_dates(start_year, end_year))
    for d in tqdm(dates, desc='Downloading WBGT'):
        fname = f'WBGT.{d.year}.{d.month:02d}.{d.day:02d}.tif'
        fpath = dest_dir / fname
        if skip_existing and fpath.exists():
            continue
        url = url_template.format(year=d.year, month=d.month, day=d.day)
        response = requests.get(url, timeout=60)
        if response.status_code == 200:
            fpath.write_bytes(response.content)
        else:
            print(f'  WARNING: {url} returned {response.status_code}')


# Uncomment to download:
# download_wbgt(EPOCH_START, EPOCH_END, WBGT_DIR)

## Compute average annual productivity loss

In [ ]:
def get_wbgt_files(wbgt_dir: Path, start_year: int, end_year: int) -> list[Path]:
    """Return sorted list of WBGT files within the epoch."""
    files = []
    for d in iter_dates(start_year, end_year):
        fname = f'WBGT.{d.year}.{d.month:02d}.{d.day:02d}.tif'
        fpath = wbgt_dir / fname
        if fpath.exists():
            files.append(fpath)
    return files


wbgt_files = get_wbgt_files(WBGT_DIR, EPOCH_START, EPOCH_END)
print(f'WBGT files found: {len(wbgt_files)}')
n_years = EPOCH_END - EPOCH_START + 1
print(f'Expected ~{n_years * 365} files for {n_years} years')

In [ ]:
# Read reference metadata from first file
with rasterio.open(wbgt_files[0]) as src:
    ref_meta = src.meta.copy()
    ref_meta.update(dtype='float32', nodata=np.nan)
    shape = (src.height, src.width)
    nodata_val = src.nodata

print(f'Grid shape: {shape}')
print(f'Resolution: {ref_meta["transform"].a:.4f}°')

In [ ]:
# Accumulate productivity loss across all days, then average
# Memory-efficient: accumulate in float64, write float32 at the end

accumulator = np.zeros(shape, dtype=np.float64)
valid_count = np.zeros(shape, dtype=np.int32)

for fpath in tqdm(wbgt_files, desc='Processing WBGT'):
    with rasterio.open(fpath) as src:
        wbgt = src.read(1).astype(np.float32)
        if nodata_val is not None:
            wbgt[wbgt == nodata_val] = np.nan

    loss = wbgt_to_productivity_loss(wbgt)  # NaN preserved

    valid = ~np.isnan(loss)
    accumulator[valid] += loss[valid]
    valid_count[valid] += 1

# Average (days with no data are excluded from denominator)
mean_annual_loss = np.where(valid_count > 0, accumulator / valid_count, np.nan).astype(np.float32)

print(f'\nProductivity loss range: {np.nanmin(mean_annual_loss):.4f} – {np.nanmax(mean_annual_loss):.4f}')
print(f'Mean (valid cells): {np.nanmean(mean_annual_loss):.4f}')

In [ ]:
# Save output raster
with rasterio.open(output_path, 'w', **ref_meta) as dst:
    dst.write(mean_annual_loss, 1)

print(f'Saved: {output_path}')

## Quick visual check

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(mean_annual_loss, cmap='YlOrRd', vmin=0, vmax=mean_annual_loss[~np.isnan(mean_annual_loss)].max())
plt.colorbar(im, ax=ax, label='Mean annual productivity loss (fraction)')
ax.set_title(f'Average Annual Productivity Loss — {EPOCH_START}–{EPOCH_END}')
plt.tight_layout()
plt.show()